# Act 2 · A Hall effect without magnetisation
*Slides [15–22](https://yaroslav-zhumagulov.github.io/demos/nm4qm-2026/slides/#a2)*

MnTe is an antiferromagnet: the moments of its two Mn sublattices cancel. Still, it shows an anomalous Hall effect
(Gonzalez Betancourt et al. 2023). The reason: the two sublattices are related by a rotation, not by a translation,
so MnTe is an altermagnet, with spin-split bands even without spin–orbit coupling. Three questions for this notebook:

1. Can one Wannierisation give both spin channels?
2. For which directions of the Néel vector is σ<sub>xy</sub> allowed?
3. How large is it, and how does it change when the Néel vector turns?

About 4 minutes of compute on 8 cores. **Before you start**, in this folder (about 45 s):
```
python 1_dft.py
```
As in Act 1: each step has a **checkpoint**, **Try it** cells take seconds, the last step is a **bonus**.

In [ ]:
import io
import inspect
import logging
import os
import sys
import warnings
from contextlib import redirect_stdout

import matplotlib.pyplot as plt
import numpy as np
from gpaw import GPAW
import gpaw.old.atomrotations
from irrep.spacegroup import SpaceGroup

import wannierberri as wb
from wannierberri.parallel import ray_init
from wannierberri.smoother import FermiDiracSmoother
from wannierberri.symmetry.projections import Projection, ProjectionsSet

if "altermagnetic" not in inspect.signature(wb.WannierDataSOC.from_gpaw).parameters:
    raise ImportError(f"this kernel ({sys.prefix}) has WannierBerri {wb.__version__}, the notebook needs its master "
                      "branch: run  bash setup/install_wannierberri.sh  there, or switch the kernel")

sys.modules["gpaw.atomrotations"] = gpaw.old.atomrotations  # irrep 3.3 looks for it where GPAW 25 had it

warnings.simplefilter("ignore")  # keep the live output readable
logging.basicConfig(level=logging.WARNING, format="%(message)s", stream=sys.stdout)
logging.getLogger("wannierberri").setLevel(logging.ERROR)  # its warnings would flood the output,
logging.getLogger("wannierberri.wannierisation").setLevel(logging.INFO)  # but show the Wannierisation progress
os.makedirs("wannier", exist_ok=True)
os.makedirs("results", exist_ok=True)
np.random.seed(0)  # the Wannierisation draws random numbers: fix them for reproducible output

## Step 1 · Symmetry of one spin channel
*Slides [16–17](https://yaroslav-zhumagulov.github.io/demos/nm4qm-2026/slides/#s11)*

The magnetic moments make Mn1 (↑) and Mn2 (↓) different species, so each spin channel
only has the 12 operations that keep the sublattices: $P\bar 3m1$.
The operations that swap them (a 60° turn about $c$ with a shift by $c/2$, and the $c$-glide) also flip the spin: that is the altermagnetic symmetry.

**Checkpoint:** moments ±4.41 μ<sub>B</sub> on Mn, 24 operations (12, doubled by time reversal), 24 irreducible k-points.

In [ ]:
calc = GPAW("gpaw/nscf.gpw", txt=None, legacy_gpaw=True)
EF = calc.get_fermi_level()
sg = SpaceGroup.from_gpaw(calc)
print("magnetic moments:", np.round(calc.get_magnetic_moments(), 2))
print(f"spin-channel group {sg.name}: {sg.size} operations (with time reversal)")
print(f"{len(calc.get_ibz_k_points())} irreducible k-points of the 6x6x4 grid (144 points)")

## Step 2 · Wannier functions for spin ↑ only
*Slide [17](https://yaroslav-zhumagulov.github.io/demos/nm4qm-2026/slides/#s12)*

13 functions: Mn1-$d$ and Te $sp^2 + p_z$ (i.e. Te $s+p$). They span all occupied spin-↑ bands
above the Mn semicore states (`IBstart=8`). With `altermagnetic=True` the spin-↓ channel is never Wannierised:
it is generated by the operation that maps Mn1 → Mn2 and flips the spin.

The NSCF grid is the SCF grid, 6×6×4 (24 irreducible points).

**Checkpoint:** after 200 iterations (about 25 s) the spreads are 0.45–0.48 Å² (Mn-d), 2.10 Å² (Te sp²) and
3.42 Å² (Te p<sub>z</sub>).

In [ ]:
Te = [[1 / 3, 2 / 3, 1 / 4], [2 / 3, 1 / 3, 3 / 4]]
projections = ProjectionsSet([
    Projection(position_num=[0, 0, 0], orbital="d", spacegroup=sg),
    Projection(position_num=Te, orbital="sp2", spacegroup=sg, xaxis=[0, -1, 0], rotate_basis=True),
    Projection(position_num=Te, orbital="pz", spacegroup=sg),
])
wandata = wb.WannierDataSOC.from_gpaw(calc, projections=projections, altermagnetic=True,
                                      IBstart=8, seedname="wannier/mnte")
wandata.wannierise(froz_min=-np.inf, froz_max=EF + 0.1, num_iter=200,
                   print_progress_every=100, sitesym=True)

In [ ]:
with redirect_stdout(io.StringIO()):  # irrep prints every symmetry operation here
    system = wb.SystemSOC.from_wannierdata(wandata, berry=True)

## Step 3 · Altermagnetic spin splitting (no SOC)
*Slide [18](https://yaroslav-zhumagulov.github.io/demos/nm4qm-2026/slides/#s13)*

Spin ↑ and ↓ bands are degenerate in the $k_z=0$ plane, but split at $k_z\neq 0$,
with opposite sign for opposite in-plane directions: a $g$-wave altermagnet.

**Checkpoint:** the two spins differ by up to 0.97 eV on the $k_z = 0.35$ Å⁻¹ part of the path.

In [ ]:
c = system.real_lattice[2, 2]
kz = 0.35 * c / (2 * np.pi)  # k_z = 0.35 1/Angstrom, in reduced units
path = wb.Path.from_nodes(system, nodes=[[2 / 3, -1 / 3, 0], [0, 0, 0], [-2 / 3, 1 / 3, 0], None,
                                         [-1 / 2, 0, kz], [0, 0, kz], [1 / 2, 0, kz]],
                          labels=["K", "Γ", "K", r"$\overline{\mathrm{M}}$", r"$\overline{\Gamma}$", r"$\overline{\mathrm{M}}$"],
                          length=300)   # barred labels: the k_z = 0.35 1/Angstrom plane
E_up = wb.evaluate_k_path(system.system_up, path=path).get_eigenvalues() - EF
E_dn = wb.evaluate_k_path(system.system_down, path=path).get_eigenvalues() - EF
x = path.getKline()
ticks, names = [], []
for i, name in path.labels.items():
    if ticks and np.isclose(x[i], ticks[-1]):
        names[-1] += "|" + name
    else:
        ticks.append(x[i])
        names.append(name)
print(f"max |E_up - E_dn|: {np.abs(E_up - E_dn).max():.3f} eV")

fig, ax = plt.subplots(figsize=(7, 4))
up = ax.plot(x, E_up, color="C3", lw=1.2)
down = ax.plot(x, E_dn, color="C0", lw=1.2, ls="--")
ax.legend([up[0], down[0]], ["spin ↑", "spin ↓"], loc="upper center", ncol=2)   # one entry per spin
ax.set_xticks(ticks, names)
ax.set_xlim(x[0], x[-1])
ax.set_ylim(-6, 0.5)
ax.set_ylabel("E − E$_F$ (eV)")
ax.set_title("no SOC")

## Step 4 · SOC on demand: choose the Néel vector
*Slide [19](https://yaroslav-zhumagulov.github.io/demos/nm4qm-2026/slides/#s14)*

`set_soc_axis(theta, phi)` rotates the spin quantisation axis $\hat{\mathbf n}$ and rebuilds the
magnetic point group used to symmetrise the results. Here $\hat{\mathbf n}\parallel y = [01\bar10]$, perpendicular to $a$.
The colour is the spin along $\hat{\mathbf n}$. In the $k_z = 0$ plane it is zero by symmetry, so those bands stay grey.

**Checkpoint:** SOC raises the highest state on this path from −0.446 to −0.317 eV.

In [ ]:
system.set_soc_axis(theta=90, phi=90, units="degrees")
tab = wb.evaluate_k_path(system, path=path, quantities=["spin"])
E_soc, S = tab.get_eigenvalues() - EF, tab.get_data("spin")
print(f"highest state on this path: {E_up.max():.3f} eV without SOC, {E_soc.max():.3f} eV with SOC")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, E_soc, color="0.8", lw=0.8)
sc = ax.scatter(np.repeat(x, E_soc.shape[1]), E_soc.ravel(), c=S[:, :, 1].ravel(), s=3,
                cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(sc, label="spin along n,  ⟨σ$_y$⟩")
ax.set_xticks(ticks, names)
ax.set_xlim(x[0], x[-1])
ax.set_ylim(-6, 0.5)
ax.set_ylabel("E − E$_F$ (eV)")
ax.set_title("with SOC, n ∥ y")
np.savez("results/mnte_bands.npz", x=x, ticks=ticks, labels=names,
         E_up=E_up, E_dn=E_dn, E_soc=E_soc, S=S)

**Check against GPAW.** GPAW's second-variation SOC (all 2 × 40 bands of the collinear run) is the reference for the
model's first-order SOC. It takes seconds.

**Checkpoint:** 322 meV (model) vs 321 meV (GPAW) for n ∥ c, 75 vs 60 meV for n ∥ y.

In [ ]:
from gpaw.spinorbit import soc_eigenstates

kibz = calc.get_ibz_k_points()
levels = {"Energy": wb.calculators.tabulate.Energy(degen_thresh=1e-6)}
iA = np.argmin(np.linalg.norm(kibz - [0, 0, 0.5], axis=1))   # the A point
E_A0 = calc.get_eigenvalues(kpt=iA, spin=0)[20]              # top valence level at A without SOC
for name, (theta, phi) in {"n || c": (0, 0), "n || y": (90, 90)}.items():
    system.set_soc_axis(theta=theta, phi=phi, units="degrees")
    E_w = wb.evaluate_k(system, k=kibz[iA], calculators={"E": levels["Energy"]}).data[0].max()
    E_g = np.sort(soc_eigenstates(calc, theta=theta, phi=phi).eigenvalues()[iA])[41]
    print(f"{name}: SOC lifts the top level at A by {(E_w - E_A0) * 1e3:4.0f} meV (model), "
          f"{(E_g - E_A0) * 1e3:4.0f} meV (GPAW)")

**Rotate the Néel vector** from c to y in 10° steps, one `set_soc_axis` call per angle, no new DFT.
We keep the spin component along $\hat{\mathbf n}$ for the plots.

**Checkpoint:** the shift at A falls smoothly from 322 to 75 meV.

In [ ]:
thetas = np.arange(0, 91, 10)
E_rot, S_rot, shift_A = [], [], []
for theta in thetas:
    system.set_soc_axis(theta=theta, phi=90, units="degrees")
    n = np.array([0, np.sin(np.deg2rad(theta)), np.cos(np.deg2rad(theta))])   # Neel vector in the y-z plane
    tab = wb.evaluate_k_path(system, path=path, quantities=["spin"])
    E_rot.append(tab.get_eigenvalues() - EF)
    S_rot.append(tab.get_data("spin") @ n)
    E_top = wb.evaluate_k(system, k=kibz[iA], calculators={"E": levels["Energy"]}).data[0].max()
    shift_A.append((E_top - E_A0) * 1e3)
    print(f"theta = {theta:2d} deg from c:  SOC lifts the top level at A by {shift_A[-1]:4.0f} meV")
np.savez("results/mnte_soc_rotation.npz", theta=thetas, x=x, E=np.array(E_rot), S_n=np.array(S_rot),
         shift_A=np.array(shift_A))

**Try it.** Now turn $\hat{\mathbf n}$ inside the basal plane (θ = 90°, φ from the a axis). Does the shift at A
change? Predict, then run.

In [ ]:
for phi in (0, 30, 60, 90):
    system.set_soc_axis(theta=90, phi=phi, units="degrees")
    E_top = wb.evaluate_k(system, k=kibz[iA], calculators={"E": levels["Energy"]}).data[0].max()
    print(f"phi = {phi:2d} deg from a:  SOC lifts the top level at A by {(E_top - E_A0) * 1e3:4.0f} meV")

## Step 5 · Anomalous Hall conductivity for three Néel vectors
*Slide [20](https://yaroslav-zhumagulov.github.io/demos/nm4qm-2026/slides/#s15)*

$$\sigma_{xy} = -\frac{e^2}{\hbar}\int\frac{d^3k}{(2\pi)^3}\sum_n f(\varepsilon_{n\mathbf k})\,\Omega^z_n(\mathbf k),
\qquad \Omega^z_n = -2\,\mathrm{Im}\sum_{m\ne n}\frac{\langle n|\partial_{k_x}H|m\rangle\langle m|\partial_{k_y}H|n\rangle}{(\varepsilon_n-\varepsilon_m)^2}$$
Symmetry allows a Hall vector along $c$ only for $\hat{\mathbf n}\parallel y = [01\bar10]$;
for $\hat{\mathbf n}\parallel x = [2\bar1\bar10]$ (the $a$ axis) and $\hat{\mathbf n}\parallel c$ it must vanish.

* Grids (multiples of the 6×6×4 Wannier grid): 96×96×64 for $\hat{\mathbf n}\parallel y$, about 40 s on 8 cores.
  At 150 K it is within ~12 S/cm of a 256×256×170 grid over the whole range (optional checks in `03_checks/`); 48×48×32 is
  4 times faster but its peaks can be off by 50% or more. For $\hat{\mathbf n}\parallel x$ and $\hat{\mathbf n}\parallel c$
  symmetry makes $\sigma_{xy}$ vanish on any grid, so the coarse one (~10 s) shows it.
* `degen_thresh=1e-4`: bands closer than 0.1 meV are treated as one group. Larger thresholds (1 meV and up)
  merge real SOC-split pairs and change the result near the band edge; 10⁻⁵ gives the same numbers as 10⁻⁴.
* Occupation: Fermi–Dirac at 150 K, the temperature of the Hall measurement of Gonzalez Betancourt et al. (2023)
  ($T_N$ = 310 K). The model has the fully ordered moments of the DFT run.
* Hole density from the cumulative DOS in the rigid-band picture, $p(E_F) = [N_{\rm W} - N(E_F)]/V_{\rm cell}$,
  with $N_{\rm W} = 26$ spin orbitals per cell, all filled in the undoped crystal.

**Checkpoint:** zero for n ∥ x and n ∥ z, up to 194 S/cm for n ∥ y (about 1 min).

In [ ]:
try:
    ray_init(logging_level="error", log_to_driver=False)   # k-points in parallel on all cores, quietly
except ImportError:
    print("Ray is not installed: running in serial, several times slower")
T = 150  # K
Efermi = np.linspace(EF - 1.2, EF + 0.2, 1401)
smoother = FermiDiracSmoother(Efermi, T_Kelvin=T)
calculators = {"ahc": wb.calculators.static.AHC(Efermi=Efermi, degen_thresh=1e-4, smoother=smoother),
               "cumdos": wb.calculators.static.CumDOS(Efermi=Efermi, smoother=smoother)}
directions = {"x": (90, 0), "y": (90, 90), "z": (0, 0)}   # x = [2-1-10] (a), y = [01-10], z = c
grids = {"x": (48, 48, 32), "y": (96, 96, 64), "z": (48, 48, 32)}
sigma, holes, results = {}, {}, {}
volume = abs(np.linalg.det(system.real_lattice)) * 1e-24   # cm^3
for name, (theta, phi) in directions.items():
    system.set_soc_axis(theta=theta, phi=phi, units="degrees")
    result = wb.run(system, grid=wb.Grid(system, NK=grids[name]), calculators=calculators,
                    fout_name="wannier/mnte", print_progress_step_time=60)
    results[name] = result
    sigma[name] = result.results["ahc"].dataSmooth / 100  # S/cm, Hall vector (x, y, z)
    holes[name] = (system.num_wann - result.results["cumdos"].dataSmooth) / volume  # holes per cm^3
    print(f"n || {name}: max |sigma| = {np.round(np.abs(sigma[name]).max(axis=0), 1)} S/cm")

Where is the valence band top with SOC? It is not at a high-symmetry point: the highest band has several close
maxima, so we search a 24×24×13 grid and zoom into the best six candidates.

**Checkpoint:** $E_v - E_F^0 = -0.314$ eV (about 15 s).

In [ ]:
energy = {"Energy": wb.calculators.tabulate.Energy(degen_thresh=1e-6)}


def top_valence(k_red):
    tab = wb.evaluate_k_path(system, path=wb.Path(system, k_list=k_red), tabulators=energy, parallel=False)
    return tab.get_eigenvalues()[:, system.num_wann - 1]


system.set_soc_axis(theta=90, phi=90, units="degrees")
k = np.stack(np.meshgrid(np.arange(24) / 24, np.arange(24) / 24, np.linspace(0, 0.5, 13), indexing="ij"), -1)
k = k.reshape(-1, 3)
E_top = np.concatenate([top_valence(k[i:i + 500]) for i in range(0, len(k), 500)])
Ev, k_v = -np.inf, None
for k0 in k[np.argsort(E_top)[::-1][:6]]:
    step = 1 / 24
    for _ in range(4):
        d = np.stack(np.meshgrid(*[np.linspace(-step, step, 7)] * 3, indexing="ij"), -1).reshape(-1, 3)
        E0 = top_valence(k0 + d)
        k0, step = k0 + d[E0.argmax()], step / 3
    if E0.max() - EF > Ev:
        Ev, k_v = E0.max() - EF, k0
print(f"valence band top with SOC, n || y: E_v - E_F0 = {Ev:+.3f} eV at k = {np.round(k_v, 3)} (reduced)")

For comparison: the films of Gonzalez Betancourt et al. have $p = 4.9\cdot10^{18}$ cm⁻³. At 150 K this puts $E_F$ about
10 meV above the band top (the Fermi tail still leaves holes), where $\sigma_{xy}$ is below 1 S/cm; they estimated
~1 S/cm from DFT on a 400³ grid.

In [ ]:
E_rel = Efermi - EF - Ev   # E_F - E_v
fig, ax = plt.subplots(figsize=(6, 4))
for name, ls in zip(directions, ("--", "-", ":")):   # x and z both give zero: dashed and dotted
    ax.plot(E_rel, sigma[name][:, 2], ls=ls, label=f"n ∥ {name}")
ax.axvline(0, color="0.7", lw=0.8)
ax.set_xlabel("E$_F$ − E$_v$ (eV)")
ax.set_ylabel("σ$_{xy}$ (S/cm)")
ax.legend()
np.savez("results/mnte_ahc.npz", E=Efermi - EF, Ev=Ev, T=T, **{f"sigma_{k}": v for k, v in sigma.items()},
         **{f"holes_{k}": v for k, v in holes.items()}, **{f"NK_{k}": v for k, v in grids.items()})

**Try it.** Temperature only changes the occupation, so the n ∥ y result can be re-smoothed without new k-points.
How much does 300 K change σ<sub>xy</sub>, and what does the unsmoothed (T = 0) curve look like?

In [ ]:
raw = results["y"].results["ahc"].data[:, 2] / 100          # T = 0, S/cm
i4 = np.argmin(np.abs(E_rel + 0.4))                         # E_F = E_v - 0.4 eV
for T_try in (0, 150, 300):
    s_T = FermiDiracSmoother(Efermi, T_Kelvin=T_try)(raw) if T_try else raw
    print(f"T = {T_try:3d} K:  sigma_xy(E_v - 0.4 eV) = {s_T[i4]:6.1f} S/cm,  "
          f"largest |sigma_xy| = {np.abs(s_T).max():6.1f} S/cm")

## Step 6 · Rotating the Néel vector in the basal plane
*Slides [21–22](https://yaroslav-zhumagulov.github.io/demos/nm4qm-2026/slides/#s16)*

No new DFT: only `set_soc_axis` changes. Symmetry (the 180° turn about a, and the 60° turn about c with a shift by c/2 combined with time reversal) makes
$\sigma_{xy}$ odd in $\varphi$ and antiperiodic with 60°:
$$\sigma_{xy}(-\varphi) = -\sigma_{xy}(\varphi),\qquad \sigma_{xy}(\varphi+60^\circ) = -\sigma_{xy}(\varphi)
\quad\Longrightarrow\quad \sigma_{xy}(\varphi) = A_3\sin 3\varphi + A_9\sin 9\varphi + A_{15}\sin 15\varphi + \dots$$
First the two relations, on the live grid. Each angle is a separate run with its own magnetic point group,
so they test the symmetry of the Wannier model itself.

**Checkpoint:** both sums are about 10⁻⁷ S/cm (about 35 s).

In [ ]:
ahc = {"ahc": calculators["ahc"]}
s = {}
for phi in (10, -10, 70):
    system.set_soc_axis(theta=90, phi=phi, units="degrees")
    s[phi] = wb.run(system, grid=wb.Grid(system, NK=(48, 48, 32)), calculators=ahc,
                    fout_name="wannier/mnte", print_progress_step_time=60).results["ahc"].dataSmooth[:, 2] / 100
print(f"max |sigma(10) + sigma(-10)| = {np.abs(s[10] + s[-10]).max():.1e} S/cm")
print(f"max |sigma(10) + sigma(70)|  = {np.abs(s[10] + s[70]).max():.1e} S/cm")

Now the amplitudes. The wedge 0°–30° holds all independent information. We read $\sigma_{xy}$ at
$E_F = E_v - 0.7$ eV, deep in the valence band, where this 48×48×32 grid agrees with a 144×144×96 grid to 4% at
every angle (optional checks in `03_checks/`). Closer to the band top, small angles need much denser grids.
About 1 minute on 8 cores.

**Checkpoint:** σ(5°)/σ(30°) = 0.56, far from the 0.26 of a pure sin 3φ; the three-harmonic fit gives
A<sub>3</sub> = 172, A<sub>9</sub> = 30 and A<sub>15</sub> = 7 S/cm.

In [ ]:
phis = np.arange(0, 30.1, 5)
E0 = Ev - 0.7  # E_F - E_F^0 in eV
i0 = np.argmin(np.abs(Efermi - EF - E0))
sigma_phi = []
for phi in phis:
    system.set_soc_axis(theta=90, phi=phi, units="degrees")
    result = wb.run(system, grid=wb.Grid(system, NK=(48, 48, 32)), calculators=ahc,
                    fout_name="wannier/mnte", print_progress_step_time=60)
    sigma_phi.append(result.results["ahc"].dataSmooth[:, 2] / 100)
    print(f"phi = {phi:4.1f}:  sigma_xy = {sigma_phi[-1][i0]:8.2f} S/cm")
sigma_phi = np.array(sigma_phi)
y = sigma_phi[:, i0]
print(f"E_F = E_v - 0.7 eV, {holes['y'][i0]:.1e} holes/cm^3")
print(f"sigma(5)/sigma(30) = {y[1] / y[-1]:.2f}; sin 3phi alone gives {np.sin(np.deg2rad(15)):.2f}")

for ms in ([3], [3, 9], [3, 9, 15]):
    M = np.sin(np.deg2rad(np.outer(phis, ms)))
    coef, *_ = np.linalg.lstsq(M, y, rcond=None)
    rms = np.sqrt(np.mean((M @ coef - y) ** 2))
    print(f"harmonics {ms}: A = {np.round(coef, 2)} S/cm,  rms residual {rms:.2f} S/cm")

fig, ax = plt.subplots(figsize=(6, 4))
p = np.linspace(0, 180, 361)
ax.plot(p, sum(a * np.sin(np.deg2rad(m * p)) for a, m in zip(coef, [3, 9, 15])), color="0.6",
        label="3-harmonic fit")
ax.plot(p, y[-1] * np.sin(np.deg2rad(3 * p)), color="0.6", ls="--", label="sin 3φ, through 30°")
ax.plot(phis, y, "o", label="computed (0°–30°)")
ax.legend()
ax.set_xlabel("φ (degrees from the a axis)")
ax.set_ylabel("σ$_{xy}$ (S/cm)")
np.savez("results/mnte_neel_scan.npz", phi=phis, E=Efermi - EF, Ev=Ev, sigma=sigma_phi, E0=E0, p_E0=holes["y"][i0])

**Try it.** The two symmetry relations give σ(60° − φ) = σ(φ), so σ(45°) should equal σ(15°) without a new
calculation. Check it (about 10 s).

In [ ]:
system.set_soc_axis(theta=90, phi=45, units="degrees")
s45 = wb.run(system, grid=wb.Grid(system, NK=(48, 48, 32)), calculators=ahc,
             fout_name="wannier/mnte", print_progress_step_time=60).results["ahc"].dataSmooth[i0, 2] / 100
print(f"sigma(45 deg) = {s45:.2f} S/cm,  sigma(15 deg) = {y[3]:.2f} S/cm")

## Bonus 7 · Where the altermagnetic splitting vanishes
*Slides [16](https://yaroslav-zhumagulov.github.io/demos/nm4qm-2026/slides/#s11), [27](https://yaroslav-zhumagulov.github.io/demos/nm4qm-2026/slides/#b2)*

A spin-flipping operation $g$ gives $E_\uparrow(\mathbf k) = E_\downarrow(g\mathbf k)$, so the splitting vanishes on every plane
that $g$ maps onto itself: $k_z = 0$ and $k_z = \pi/c$ (the mirror $m_z$ at $z=\tfrac14$), and the three vertical planes
through Γ–K (the $c$-glides). Four nodal planes make MnTe a $g$-wave altermagnet.
The map shows $E_\uparrow - E_\downarrow$ of the top valence band at $k_z = 0.35$ Å⁻¹ (no SOC).

In [ ]:
B = system.recip_lattice


def splitting(k_cart, batch=400):
    """E_up - E_down for all bands at Cartesian k-points (in batches, to keep memory low)."""
    out = []
    for i in range(0, len(k_cart), batch):
        path_k = wb.Path(system, k_list=k_cart[i:i + batch] @ np.linalg.inv(B))
        E_u = wb.evaluate_k_path(system.system_up, path=path_k, parallel=False).get_eigenvalues()
        E_d = wb.evaluate_k_path(system.system_down, path=path_k, parallel=False).get_eigenvalues()
        out.append(E_u - E_d)
    return np.concatenate(out)


K = np.linalg.norm(B[0] + B[1]) / 3
kx, ky = np.meshgrid(np.linspace(-K, K, 121), np.linspace(-K, K, 121))
edge = K * np.cos(np.pi / 6) / np.cos(np.arctan2(ky, kx) % (np.pi / 3) - np.pi / 6)
inside = np.hypot(kx, ky) <= edge + 1e-9          # hexagonal zone, corners (K points) at 0, 60, ... degrees
dE = splitting(np.stack([kx[inside], ky[inside], np.full(inside.sum(), 0.35)], axis=1))
top = np.full(kx.shape, np.nan)
top[inside] = dE[:, -1]

kzA = np.pi / system.real_lattice[2, 2]       # k_z of the A point
sub = inside & (np.arange(inside.size).reshape(inside.shape) % 3 == 0)
kpar, kperp = np.meshgrid(np.linspace(-K, K, 31), np.linspace(-kzA, kzA, 21))
planes = {"kz = 0": np.stack([kx[sub], ky[sub], np.zeros(sub.sum())], axis=1),
          "kz = pi/c": np.stack([kx[sub], ky[sub], np.full(sub.sum(), kzA)], axis=1),
          "vertical, through Gamma-K": np.stack([kpar.ravel(), np.zeros(kpar.size), kperp.ravel()], axis=1)}
for name, pts in planes.items():
    print(f"{name:26s} plane: max |E_up - E_dn| = {np.abs(splitting(pts)).max():.1e} eV over {len(pts)} points")
k = np.linspace(0, K * np.cos(np.pi / 6), 41)
line = np.stack([k * np.cos(np.pi / 6), k * np.sin(np.pi / 6), np.full_like(k, 0.35)], axis=1)
print(f"Gamma-M line at kz = 0.35 1/A: max |E_up - E_dn| = {np.abs(splitting(line)).max():.2f} eV")

fig, ax = plt.subplots(figsize=(5, 4.3))
im = ax.pcolormesh(kx, ky, top, cmap="coolwarm", vmin=-0.5, vmax=0.5, shading="auto")
fig.colorbar(im, label="E$_\\uparrow$ − E$_\\downarrow$, top valence band (eV)")
ax.set_aspect("equal")
ax.set_xlabel("k$_x$ (1/Å)")
ax.set_ylabel("k$_y$ (1/Å)")
np.savez("results/mnte_splitting_map.npz", kx=kx, ky=ky, top=top, K=K, kz=0.35)

The same map from $k_z = 0$ to $k_z = \pi/c$ in steps of 0.05 Å⁻¹ (about 20 s): the splitting grows from zero,
peaks, and vanishes again on the $k_z = \pi/c$ plane.

In [ ]:
kx_s, ky_s = np.meshgrid(np.linspace(-K, K, 61), np.linspace(-K, K, 61))
edge_s = K * np.cos(np.pi / 6) / np.cos(np.arctan2(ky_s, kx_s) % (np.pi / 3) - np.pi / 6)
inside_s = np.hypot(kx_s, ky_s) <= edge_s + 1e-9
kz_list = np.append(np.arange(0, kzA - 1e-6, 0.05), kzA)
maps = np.full((len(kz_list),) + kx_s.shape, np.nan)
for i, kz in enumerate(kz_list):
    maps[i][inside_s] = splitting(np.stack([kx_s[inside_s], ky_s[inside_s],
                                            np.full(inside_s.sum(), kz)], axis=1))[:, -1]
    print(f"kz = {kz:.3f} 1/A: max |E_up - E_dn| of the top band = {np.nanmax(np.abs(maps[i])):.2f} eV")
np.savez("results/mnte_splitting_kz.npz", kx=kx_s, ky=ky_s, kz=kz_list, maps=maps, K=K)